# BikeStore - Silver Layer Cleaning
BikeStore - Silver layer: clean Bronze CSV data and save as Delta  
Steps: snake_case columns, trim/collapse spaces, empty -> null,  
remove duplicates, drop rows with null key columns  


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import re

BASE   = "abfss://bikestore@olympicdata18.dfs.core.windows.net"
BRONZE = f"{BASE}/bronze"
SILVER = f"{BASE}/silver"

def to_snake_case(name: str) -> str:
    name = name.strip()
    name = re.sub(r"[\s\-\.]+", "_", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_").lower()

def read_table(path: str):
    return (spark.read.format("csv")
            .option("header", True)
            .option("inferSchema", True)
            .load(path))

def clean_dataframe(df, required_cols):
    # 1. snake_case column names
    for c in df.columns:
        df = df.withColumnRenamed(c, to_snake_case(c))

    # 2. trim, collapse extra spaces, turn empty / "NULL" text into real nulls
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            col = F.trim(F.regexp_replace(F.col(field.name), r"\s+", " "))
            df = df.withColumn(field.name, col)
            df = df.withColumn(
                field.name,
                F.when(F.upper(F.col(field.name)).isin("", "NULL"), None)
                 .otherwise(F.col(field.name)))

    # 3. remove duplicate rows
    df = df.dropDuplicates()

    # 4. remove rows where a KEY column is null (other columns may stay null)
    df = df.dropna(subset=required_cols)
    return df

In [ ]:
# table -> (schema folder, key columns that must not be null)
tables = {
    "customers":   ("sales",      ["customer_id", "first_name", "last_name"]),
    "orders":      ("sales",      ["order_id", "customer_id", "order_date"]),
    "order_items": ("sales",      ["order_id", "product_id", "quantity", "list_price"]),
    "staffs":      ("sales",      ["staff_id", "first_name", "last_name"]),
    "stores":      ("sales",      ["store_id", "store_name"]),
    "brands":      ("production", ["brand_id", "brand_name"]),
    "categories":  ("production", ["category_id", "category_name"]),
    "products":    ("production", ["product_id", "product_name", "brand_id", "category_id", "list_price"]),
    "stocks":      ("production", ["store_id", "product_id", "quantity"]),
}

for table, (schema, keys) in tables.items():
    df_raw   = read_table(f"{BRONZE}/{schema}/{table}/")
    df_clean = clean_dataframe(df_raw, keys)

    (df_clean.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .save(f"{SILVER}/{schema}/{table}/"))

    r, c = df_raw.count(), df_clean.count()
    print(f"{schema}.{table:<12} read={r:<6} written={c:<6} removed={r-c}")